# QUANTSTATS — Análisis profesional de resultados

## Cómo exportar las operaciones desde TradingView:
1. Abre tu estrategia en TradingView
2. Ve al Strategy Tester → pestaña **'Lista de operaciones'**
3. Haz clic en el icono de descarga (esquina superior derecha del tester)
4. Guarda el CSV en: `C:/Users/alber/tradingview-scripts/datos/operaciones.csv`

## Sin CSV: usa datos reales de Yahoo Finance (modo demo)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import yfinance as yf
import quantstats as qs
import os

# ── CONFIGURACION ──────────────────────────────────────────
SIMBOLO    = 'SPY'          # benchmark (cambia por el activo que operas)
INICIO     = '2019-01-01'
FIN        = '2026-05-01'
CSV_PATH   = r'C:\Users\alber\tradingview-scripts\datos\operaciones.csv'
OUTPUT     = r'C:\Users\alber\tradingview-scripts\analisis\reporte_quantstats.html'
os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)

In [ ]:
# ── OPCION A: cargar CSV exportado de TradingView ──────────
# TradingView exporta columnas: Trade #, Type, Signal, Date/Time, Price, Contracts, Profit, etc.

def cargar_csv_tradingview(path):
    df = pd.read_csv(path)
    # Filtrar solo filas de cierre (Entry & Exit o Exit)
    df.columns = df.columns.str.strip()
    # Buscar columna de fecha y profit
    fecha_col  = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
    profit_col = [c for c in df.columns if 'profit' in c.lower()][0]
    df[fecha_col] = pd.to_datetime(df[fecha_col])
    df = df.dropna(subset=[profit_col])
    retornos = df.set_index(fecha_col)[profit_col]
    retornos = retornos[retornos != 0]
    # Convertir a retornos porcentuales
    capital = 10000
    pct = retornos / capital
    return pct.sort_index()

if os.path.exists(CSV_PATH):
    print('CSV encontrado — cargando operaciones de TradingView...')
    returns = cargar_csv_tradingview(CSV_PATH)
    print(f'Operaciones cargadas: {len(returns)}')
else:
    print('CSV no encontrado — usando modo DEMO con datos reales de SPY')
    spy = yf.download(SIMBOLO, start=INICIO, end=FIN, progress=False)['Close'].squeeze()
    returns = spy.pct_change().dropna()
    returns.index = pd.to_datetime(returns.index).tz_localize(None)
    print(f'Datos demo cargados: {len(returns)} dias')

In [ ]:
# ── METRICAS CLAVE ─────────────────────────────────────────
bench = yf.download(SIMBOLO, start=INICIO, end=FIN, progress=False)['Close'].squeeze()
bench_ret = bench.pct_change().dropna()
bench_ret.index = pd.to_datetime(bench_ret.index).tz_localize(None)

returns.index = pd.to_datetime(returns.index).tz_localize(None)

print('── METRICAS PRINCIPALES ──')
print(f'Sharpe Ratio  : {qs.stats.sharpe(returns):.2f}  (>1 bueno, >2 excelente)')
print(f'Calmar Ratio  : {qs.stats.calmar(returns):.2f}  (retorno/drawdown)')
print(f'Max Drawdown  : {qs.stats.max_drawdown(returns)*100:.1f}%')
print(f'Retorno total : {qs.stats.comp(returns)*100:.1f}%')
print(f'Win Rate      : {qs.stats.win_rate(returns)*100:.1f}%')

In [ ]:
# ── REPORTE HTML COMPLETO ──────────────────────────────────
# Genera un informe con graficas, heatmap mensual, comparativa vs benchmark
qs.reports.html(returns, benchmark=bench_ret,
                title='Mi Estrategia — Analisis Completo',
                output=OUTPUT)

import webbrowser
webbrowser.open(OUTPUT)
print(f'Reporte generado y abierto: {OUTPUT}')